In [278]:
import pandas as pd
import numpy as np


# Data Loading


data = pd.read_csv("online_retail.csv")

df = pd.DataFrame(data)


highly_returned_products = df[df["Quantity"]<0].sort_values(by="Quantity")


# Data Cleaning

df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])

df = df.dropna(axis=0)

df = df.drop(df[df["Quantity"]<0].index)

df = df.drop(df[df["UnitPrice"]==0].index)



# Feature Engineering


df["Total_Price"] = df["Quantity"]*df["UnitPrice"]

df["Day"] = df["InvoiceDate"].dt.day_name()
df["Month"] = df["InvoiceDate"].dt.month
df["Year"] = df["InvoiceDate"].dt.year

spent_by_customer = df.groupby("CustomerID")["Total_Price"].sum()
total_customer_orders = df.groupby("CustomerID")["InvoiceNo"].nunique()
avg_order_value = spent_by_customer/total_customer_orders

last_order = df.groupby("CustomerID")["InvoiceDate"].max()


# Customer Segmentation


df1 = pd.DataFrame({
    "Total Spend per Customer":spent_by_customer,
    "Total Orders per Customer":total_customer_orders,
    "Average Order Value":avg_order_value
})

conditions = [
    df1["Total Spend per Customer"] >= 2000,
    df1["Total Spend per Customer"] >= 1000
]

choices = ["High Value", "Medium Value"]

df1["Customer Segment"] = np.select(conditions,choices,default = "Low Value")


# Time based Analysis


monthly_revenue = df.groupby("Month")["Total_Price"].sum().sort_index()

best_sales_days = df.groupby("Day")["Total_Price"].sum().sort_values(ascending=False)

# Product Based Analysis


most_sold = df.groupby(["StockCode","Description"])["Quantity"].sum().sort_values(ascending=False)
most_profitable = df.groupby(["StockCode","Description"])["Total_Price"].sum().sort_values(ascending=False)


# MarkDown

# The time based Analysis indicates that the months of december, november and october had the highest monthly revenue, likely due to the holiday season demand.
# The most profitable items were the items with the stock codes : 23843,22423,23166,85099B,85123A. Low-performing products will have to go through further investigation to find the cause.
# The analysis fo returned products may indicate issues such as product quality, incorrect orders or customer dissatisfaction



,Total Spend per Customer,Total Orders per Customer,Average Order Value,Customer Segment,Last Purchase
CustomerID,,,,,
12346.0,77183.60,1,77183.600000,High Value,2011-01-18 10:01:00
12347.0,4310.00,7,615.714286,High Value,2011-12-07 15:52:00
12348.0,1797.24,4,449.310000,Medium Value,2011-09-25 13:13:00
12349.0,1757.55,1,1757.550000,Medium Value,2011-11-21 09:51:00
12350.0,334.40,1,334.400000,Low Value,2011-02-02 16:01:00
...,...,...,...,...,...
18280.0,180.60,1,180.600000,Low Value,2011-03-07 09:52:00
18281.0,80.82,1,80.820000,Low Value,2011-06-12 10:53:00
18282.0,178.05,2,89.025000,Low Value,2011-12-02 11:43:00
